# Optimizing waste recycling points in Eindhoven

**Advanced Simulation (2DI66) - Assignment 2**

This notebook contains the source code, analysis, and reporting for a simulation model of the Cure Milieustraat-Lodewijkstraat waste recycling point. It models the flow of customers through the facility and the effective servicing dynamics of the system.

## Problem statement
Simulate the intricate recycling waste drop-off system to determine:
- Waiting times and queue lengths in the current state
- Evaluate the impact of having 20% more visitors to the WRP
- Suggest improvements that significantly improve performance (shortening service times, adding parking spots, creating different stations, changing routing).

## Architectural Critique & Necessary Corrections

Your proposed framework contains functional contradictions that will cause the simulation to behave incorrectly if implemented exactly as specified. Review these critical corrections:

1. **Routing Contradiction (Route 2 vs. Assumption A3):** You defined Route 2 as a strict sequence (`Entrance ⟶ Hall ⟶ Overflow ⟶ DcDd ⟶ Rest`). However, Assumption A3 dictates that a `Type B` vehicle will only visit *either* the `Hall` or `Overflow`, never both sequentially. This requires a conditional routing junction after the entrance, not a linear pathway.
2. **Parking Bay Allocation Inefficiency:** Your `Waste Zone` properties suggest tracking 'indexed parking bay pairs' to accommodate `Big` vehicles. This introduces significant and unnecessary computational overhead. The mathematically standard approach is to use a scalar capacity model: allocate the zone a total capacity of units (e.g., 12 small bays = 12 capacity units), where `Small` vehicles request 1 unit and `Big` vehicles request 2 units from the resource pool.
3. **State Machine Redundancy:** Your proposed state graph lists `Service ⟶ Waiting` twice and introduces `Traveling` without full resolution. The standard discrete event lifecycle `Arrive ⟶ Queue (Wait) ⟶ Seize (Service) ⟶ Release ⟶ Route/Depart` should be utilized to maintain strict entity state control.

In [2]:
import simpy
import random
import pandas as pd

class Customer:
    def __init__(self, env, uid, vehicle_size, waste_types):
        self.env = env
        self.uid = uid
        self.vehicle_size = vehicle_size  # 'Small' or 'Big'
        self.waste_types = waste_types    # List of waste categories
        self.capacity_cost = 1 if vehicle_size == 'Small' else 2
        
        # Metrics tracking
        self.arrival_time = env.now
        self.total_wait_time = 0.0
        self.total_service_time = 0.0

class WasteZone:
    def __init__(self, env, name, capacity, queue_limit, service_dist):
        self.env = env
        self.name = name
        # Capacity represents total small bays. Big vehicles request 2.
        self.bays = simpy.PriorityResource(env, capacity=capacity)
        self.queue_limit = queue_limit
        self.service_dist = service_dist  # Function returning a sample time
        
    def serve(self, customer):
        arrive_zone = self.env.now
        
        # Check single file queue limit (R2/A10 logic applies here)
        if len(self.bays.queue) >= self.queue_limit:
            # Logic for upstream blocking goes here
            pass 
            
        with self.bays.request(priority=0) as req:
            yield req
            wait_time = self.env.now - arrive_zone
            customer.total_wait_time += wait_time
            
            service_time = self.service_dist()
            yield self.env.timeout(service_time)
            customer.total_service_time += service_time

class WasteRecyclingPlant:
    def __init__(self, env):
        self.env = env
        # Initialization of zones based on WRP description
        self.zones = {
            'Entrance': WasteZone(env, 'Entrance', capacity=1, queue_limit=float('inf'), service_dist=lambda: random.expovariate(1/2.0)),
            'Hall': WasteZone(env, 'Hall', capacity=12, queue_limit=14, service_dist=lambda: random.triangular(5, 15, 10)),
            'Overflow': WasteZone(env, 'Overflow', capacity=10, queue_limit=float('inf'), service_dist=lambda: random.triangular(3, 10, 6)),
            'DcDd': WasteZone(env, 'DcDd', capacity=7, queue_limit=float('inf'), service_dist=lambda: random.expovariate(1/8.0)),
            'Green': WasteZone(env, 'Green', capacity=5, queue_limit=float('inf'), service_dist=lambda: random.expovariate(1/5.0)),
            'Rest': WasteZone(env, 'Rest', capacity=5, queue_limit=float('inf'), service_dist=lambda: random.expovariate(1/6.0))
        }
        self.completed_customers = []

    def route_customer(self, customer):
        # 1. All vehicles pass Entrance
        yield self.env.process(self.zones['Entrance'].serve(customer))
        
        # 2. Routing logic based on constraints (A1-A10)
        if 'Type A' in customer.waste_types:
            yield self.env.process(self.zones['Green'].serve(customer))
            if 'Rest' in customer.waste_types:
                yield self.env.process(self.zones['Rest'].serve(customer))
        
        elif 'Type B' in customer.waste_types:
            # A5 & A7: Big vehicles go to Hall. Small prefer Overflow.
            if customer.vehicle_size == 'Big':
                yield self.env.process(self.zones['Hall'].serve(customer))
            else:
                # Implementation of A7 preference logic required here
                yield self.env.process(self.zones['Overflow'].serve(customer))
            
            if 'DcDd' in customer.waste_types:
                yield self.env.process(self.zones['DcDd'].serve(customer))
            if 'Rest' in customer.waste_types:
                yield self.env.process(self.zones['Rest'].serve(customer))
                
        self.completed_customers.append(customer)

# Execution block stub
env = simpy.Environment()
plant = WasteRecyclingPlant(env)
# env.run(until=480) # 8 hours


In [3]:
env.run(until=12*6)